In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import json
from pathlib import Path
from zipfile import ZipFile, ZIP_STORED
from tqdm.auto import tqdm

In [ ]:
DRIVE = Path("/content/drive/MyDrive/WasteScope")

ARCHIVE = DRIVE / "raw_archives/detection_task.zip"
MANIFEST = DRIVE / "manifests/video_split_v1.json"

RAW = Path("/content/wastescope_raw_detection")
PREPARED = Path("/content/wastescope_detection_prepared")
OUTPUT = DRIVE / "prepared/detection_v1"

SPLITS = ("train", "val", "test")
CLASSES = [
    "container_yellow",
    "container_green",
    "container_default",
    "container_blue",
    "container_ash",
    "container_oil",
    "container_battery",
    "container_biodegradable",
]


RAW.mkdir(parents=True, exist_ok=True)
with ZipFile(ARCHIVE) as archive:
    for member in tqdm(archive.infolist(), desc="Extracting full detection ZIP"):
        archive.extract(member, RAW)

Extracting full detection ZIP:   0%|          | 0/72960 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))

video_split = {}

for split in SPLITS:
    for video in manifest["splits"][split]:
        if video in video_split:
            raise ValueError(f"Repeated video ID: {video}")

        video_split[video] = split

task_folder = RAW / "Detection Task"

images = {
    path.stem: path
    for path in task_folder.rglob("*.png")
}

labels = {
    path.stem: path
    for path in task_folder.rglob("*.txt")
}

if not images or images.keys() != labels.keys():
    raise ValueError("Image এবং label pairs মিলছে না।")

source_videos = {stem.split("_")[0] for stem in images}

if source_videos != set(video_split):
    raise ValueError("Archive এবং manifest-এর video IDs মিলছে না।")

for split in SPLITS:
    (PREPARED / split / "images").mkdir(parents=True, exist_ok=True)
    (PREPARED / split / "labels").mkdir(parents=True, exist_ok=True)

counts = {split: 0 for split in SPLITS}


for stem in tqdm(sorted(labels), desc="Preparing labels"):
    split = video_split[stem.split("_")[0]]
    converted = []

    rows = labels[stem].read_text(encoding="utf-8-sig").splitlines()

    for line_number, row in enumerate(rows, start=1):
        fields = row.split()

        if not fields:
            continue

        if len(fields) != 5 or fields[0] not in {
            str(class_id) for class_id in range(9)
        }:
            raise ValueError(
                f"Invalid label: {labels[stem]}, line {line_number}"
            )

        raw_class = int(fields[0])

        if raw_class > 0:
            converted.append(
                " ".join([str(raw_class - 1), *fields[1:]])
            )

    text = "\n".join(converted)

    if text:
        text += "\n"

    destination = PREPARED / split / "labels" / f"{stem}.txt"
    destination.write_text(text, encoding="utf-8")

for stem in tqdm(sorted(images), desc="Splitting images"):
    split = video_split[stem.split("_")[0]]
    destination = PREPARED / split / "images" / f"{stem}.png"

    images[stem].replace(destination)
    counts[split] += 1

yaml_text = (
    "train: train/images\n"
    "val: val/images\n"
    "test: test/images\n"
    "names:\n"
)

yaml_text += "".join(
    f"  {class_id}: {name}\n"
    for class_id, name in enumerate(CLASSES)
)

(PREPARED / "data.yaml").write_text(yaml_text, encoding="utf-8")
(PREPARED / "video_split_v1.json").write_bytes(MANIFEST.read_bytes())



OUTPUT.mkdir(parents=True, exist_ok=True)

for split in SPLITS:
    temporary = OUTPUT / f"{split}.zip.part"
    destination = OUTPUT / f"{split}.zip"

    files = sorted(
        path
        for path in (PREPARED / split).rglob("*")
        if path.is_file()
    )

    with temporary.open("wb", buffering=8 * 1024**2) as stream:
        with ZipFile(stream, "w", compression=ZIP_STORED) as archive:
            for name in ("data.yaml", "video_split_v1.json"):
                archive.write(PREPARED / name, arcname=name)

            for path in tqdm(files, desc=f"Saving {split}.zip"):
                archive.write(
                    path,
                    arcname=path.relative_to(PREPARED).as_posix()
                )

    temporary.replace(destination)

    print(f"Saved {destination.name}: {counts[split]:,} images")

print(f"Completed: {OUTPUT}")